# Speech Commands Classification Pipeline: 1D ConvNeXt with Channel Attention and Multi-Head Temporal Pooling

This notebook implements an end-to-end audio classification pipeline for Keyword Spotting (KWS) on the Google Speech Commands dataset (`torchaudio.datasets.SPEECHCOMMANDS`) using a 1D ConvNeXt neural architecture with squeeze-and-excitation channel attention and multi-head temporal attention pooling.

The code is organized into sequential functional stages:
1. Environment setup and dependency installation
2. Telemetry dashboard extension
3. Core library imports and environment setup
4. Compute device allocation
5. Hyperparameter and directory configuration
6. Acoustic feature extraction and augmentation operators
7. Dataset ingestion and DataLoader instantiation
8. Exploratory acoustic inspection and audio playback
9. Neural network architecture definition
10. Model instantiation and parameter summary
11. Loss function and learning rate schedule
12. Training and validation loop execution
13. Quantitative evaluation, confusion analysis, and error diagnostics
14. Interactive audio inference and prediction pipeline

## 1. Environment Setup and Dependency Installation

Install all required runtime dependencies including core deep learning libraries, accelerator backends, telemetry tooling, and audio processing utilities.

In [ ]:
%pip install -q torch torchvision torchaudio torch_xla tensorboard matplotlib pandas ipywidgets scipy torchmetrics safetensors

## 2. Telemetry Dashboard Extension

Load the inline telemetry viewer to inspect training progression, loss/accuracy curves, logged spectrograms, and computational graph structures directly within the execution environment.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports and Environment Setup

Import standard numerical processing, audio signal processing, deep learning modules, accelerator runtime interfaces, and visualization interfaces. Set global random seeds for deterministic execution.

In [ ]:
import json
import random
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchaudio
from torchaudio.datasets import SPEECHCOMMANDS
import torchaudio.transforms as T
import IPython.display as ipd
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | TorchAudio Version: {torchaudio.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)
        if loss_fn is not None and len(loss_fn.state_dict()) > 0:
            loss_dict = {
                k: v.detach().cpu().contiguous()
                for k, v in loss_fn.state_dict().items()
            }
            zf.writestr("loss_fn.safetensors", safetensors_save(loss_dict))


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
    loss_fn: torch.nn.Module | None = None,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")
        if loss_fn is not None and "loss_fn.safetensors" in zf.namelist():
            loss_bytes = zf.read("loss_fn.safetensors")
            loss_dict = safetensors_load(loss_bytes)
            loss_dict = {k: v.to(device) for k, v in loss_dict.items()}
            loss_fn.load_state_dict(loss_dict)

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)

    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Initialize and allocate the compute accelerator (Cloud TPU via PyTorch XLA, NVIDIA CUDA GPU, Apple Silicon MPS, or CPU).

In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"

print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Define file system locations for datasets, telemetry logs, and saved checkpoints relative to the repository root (`../../`). Set acoustic processing parameters ($16\text{ kHz}$, $80$ Mel bins) and optimization hyperparameters.

In [ ]:
# ── Directory Paths ────────────────────────────────────────────────────────
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/audio_speech_commands").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/audio").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "speech_commands_convnext.zip"

# ── Acoustic Signal Parameters ───────────────────────────────────────────
SAMPLE_RATE = 16000
MAX_DURATION_SEC = 1.0
TARGET_LENGTH = int(SAMPLE_RATE * MAX_DURATION_SEC)
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
N_MELS = 80
F_MIN = 20.0
F_MAX = 8000.0

# ── Optimization & Pipeline Hyperparameters ──────────────────────────────
BATCH_SIZE = 128
EPOCHS = 20
BASE_LR = 3e-4
MAX_LR = 1.5e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PIN_MEMORY = device.type == "cuda"
LABEL_SMOOTHING = 0.1

print(
    f"Config: Batch Size = {BATCH_SIZE} | Epochs = {EPOCHS} | Sample Rate = {SAMPLE_RATE} Hz | Mel Bins = {N_MELS}"
)

## 6. Acoustic Feature Extraction and Augmentation Operators

Assemble the acoustic feature transformation and augmentation operators:
- **Waveform Pad/Trim**: Enforces a uniform duration of $1.0\text{ s}$ ($16,000$ samples) with symmetric zero-padding or center-cropping.
- **Log Mel-Spectrogram Extraction**: Computes $80$-channel log Mel-filterbank energies ($N_{\text{fft}}=1024, \text{hop}=256, \text{win}=1024$):
  $$S_{\text{mel}}(f, t) = 10 \log_{10} \left( \max(\epsilon, \mathbf{M} \cdot |\text{STFT}(x)|^2) \right)$$
- **SpecAugment**: Applies random time masking ($\Delta t=25$) and frequency masking ($\Delta f=15$) during training to regularize acoustic feature representations.

In [ ]:
class AudioFeatureExtractor(nn.Module):
    """Extracts Log-Mel Spectrogram features with optional SpecAugment."""

    def __init__(
        self,
        sample_rate=16000,
        n_fft=1024,
        hop_length=256,
        n_mels=80,
        is_training=False,
    ):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            win_length=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=F_MIN,
            f_max=F_MAX,
            power=2.0,
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80.0)
        self.is_training = is_training

        # SpecAugment: Frequency and Time masking
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=25)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Force float32 computation outside autocast to prevent fp16 log underflow to -inf / nan
        with torch.amp.autocast(x.device.type, enabled=False):
            x_float = x.float()
            mel = self.mel_transform(x_float)
            log_mel = self.amplitude_to_db(mel)

            if self.is_training and self.training:
                log_mel = self.freq_mask(log_mel)
                log_mel = self.time_mask(log_mel)

            # Squeeze channel dim if 1: [B, N_mels, Time]
            return log_mel.squeeze(1)


def pad_or_trim(waveform: torch.Tensor, target_length: int = 16000) -> torch.Tensor:
    """Pads or center-crops waveform to exactly target_length."""
    _, length = waveform.shape
    if length < target_length:
        pad_left = (target_length - length) // 2
        pad_right = target_length - length - pad_left
        return F.pad(waveform, (pad_left, pad_right), mode="constant", value=0.0)
    elif length > target_length:
        start = (length - target_length) // 2
        return waveform[:, start : start + target_length]
    return waveform

## 7. Dataset Ingestion and DataLoader Instantiation

Download and cache the official Google Speech Commands v0.02 dataset. Build deterministic label encodings across the 35 command classes and construct multi-process training, validation, and test DataLoaders.

In [ ]:
# Initialize SpeechCommands dataset
print("Loading SpeechCommands dataset (v0.02)...")
full_dataset = SPEECHCOMMANDS(root=str(DATA_DIR), download=True, subset=None)

# Extract unique labels across the dataset
COMMAND_CLASSES = sorted(
    list(
        set(
            Path(p).parent.name
            for p in full_dataset._walker
            if not Path(p).parent.name.startswith("_")
        )
    )
)
label_to_idx = {label: idx for idx, label in enumerate(COMMAND_CLASSES)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
NUM_CLASSES = len(COMMAND_CLASSES)

print(
    f"Total Audio Samples: {len(full_dataset)} | Classes ({NUM_CLASSES}): {COMMAND_CLASSES[:10]}..."
)

# Load train, validation, and test subsets
train_set = SPEECHCOMMANDS(root=str(DATA_DIR), download=True, subset="training")
val_set = SPEECHCOMMANDS(root=str(DATA_DIR), download=True, subset="validation")
test_set = SPEECHCOMMANDS(root=str(DATA_DIR), download=True, subset="testing")

print(
    f"Splits -> Training: {len(train_set):,} | Validation: {len(val_set):,} | Testing: {len(test_set):,}"
)


def collate_speech_commands(batch):
    """Collate function that standardizes waveform length and encodes labels."""
    waveforms, labels = [], []
    for waveform, sample_rate, label, speaker_id, utterance_num in batch:
        # Standardize sample rate if needed
        if sample_rate != SAMPLE_RATE:
            resampler = T.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)
            waveform = resampler(waveform)
        # Pad or trim to 16,000 samples
        waveform = pad_or_trim(waveform, TARGET_LENGTH)
        waveforms.append(waveform)
        labels.append(label_to_idx[label])

    return torch.stack(waveforms), torch.tensor(labels, dtype=torch.long)


train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_speech_commands,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_speech_commands,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_speech_commands,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

## 8. Exploratory Acoustic Inspection and Audio Playback

Extract sample audio waveforms and compute their corresponding log Mel-spectrogram representations alongside interactive audio playback widgets.

In [ ]:
# Inspect a batch
sample_waves, sample_labels = next(iter(train_loader))
feature_extractor = AudioFeatureExtractor(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    is_training=False,
)
sample_mels = feature_extractor(sample_waves)

print(f"Waveform Batch Shape: {sample_waves.shape} (Batch, Channel, Samples)")
print(f"Log Mel-Spectrogram Shape: {sample_mels.shape} (Batch, Mels, Time Frames)")

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
for i in range(3):
    # Waveform plot
    time_axis = np.linspace(0, MAX_DURATION_SEC, TARGET_LENGTH)
    axes[i, 0].plot(time_axis, sample_waves[i, 0].numpy(), color="#1f77b4", lw=1.0)
    axes[i, 0].set_title(f"Label: '{idx_to_label[sample_labels[i].item()]}' - Waveform")
    axes[i, 0].set_ylabel("Amplitude")
    axes[i, 0].set_xlabel("Time (s)")
    axes[i, 0].grid(True, alpha=0.3)

    # Mel-Spectrogram plot
    im = axes[i, 1].imshow(
        sample_mels[i].numpy(),
        origin="lower",
        aspect="auto",
        extent=[0, MAX_DURATION_SEC, F_MIN, F_MAX],
        cmap="magma",
    )
    axes[i, 1].set_title(f"Log Mel-Spectrogram (80 bins)")
    axes[i, 1].set_ylabel("Frequency (Hz)")
    axes[i, 1].set_xlabel("Time (s)")
    fig.colorbar(im, ax=axes[i, 1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

# Interactive Audio Playback for first sample
print(f"Playing sample 0: Command '{idx_to_label[sample_labels[0].item()]}'")
ipd.Audio(sample_waves[0, 0].numpy(), rate=SAMPLE_RATE)

## 9. Neural Network Architecture Definition

Define the 1D ConvNeXt classification architecture:
- **1D Stem**: 1D convolution ($k=7, s=2$) mapping raw Mel filterbanks to embedding dimensions.
- **ConvNeXt-1D Blocks**: $7\times 1$ depthwise convolution, channels-first LayerNorm, inverted bottleneck expansion ($4\times$), GELU non-linearities, and Squeeze-and-Excitation (SE) channel recalibration.
- **Multi-Head Temporal Attention Pooling**: Aggregates temporal frame features into a fixed-dimensional utterance representation using learned multi-head attention weights.
- **Linear Classifier**: Final projection from pooled embeddings to class logits.

In [ ]:
class SqueezeExcitation1D(nn.Module):
    """Channel Attention module for 1D representations."""

    def __init__(self, dim: int, reduction: int = 4):
        super().__init__()
        reduced_dim = max(8, dim // reduction)
        self.fc1 = nn.Linear(dim, reduced_dim, bias=False)
        self.fc2 = nn.Linear(reduced_dim, dim, bias=False)
        self.act = nn.GELU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        w = x.mean(dim=-1)
        w = self.fc2(self.act(self.fc1(w)))
        w = self.sigmoid(w).unsqueeze(-1)
        return x * w


class ConvNeXt1DBlock(nn.Module):
    """1D ConvNeXt Block with Depthwise Convolution and Inverted Bottleneck."""

    def __init__(
        self, dim: int, drop_path: float = 0.0, layer_scale_init_value: float = 1e-6
    ):
        super().__init__()
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.GroupNorm(1, dim)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.se = SqueezeExcitation1D(4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = (
            nn.Parameter(
                layer_scale_init_value * torch.ones((dim, 1)), requires_grad=True
            )
            if layer_scale_init_value > 0
            else None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = x.permute(0, 2, 1)
        x = self.pwconv1(x)
        x = self.act(x)
        x = x.permute(0, 2, 1)
        x = self.se(x)
        x = x.permute(0, 2, 1)
        x = self.pwconv2(x)
        x = x.permute(0, 2, 1)

        if self.gamma is not None:
            x = self.gamma * x

        return residual + x


class MultiHeadTemporalAttentionPooling(nn.Module):
    """Multi-head attention pooling over time dimension."""

    def __init__(self, dim: int, num_heads: int = 4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.query = nn.Parameter(torch.randn(num_heads, self.head_dim) * 0.02)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        B, C, T = x.shape
        x_proj = self.proj(x.permute(0, 2, 1))
        x_reshaped = x_proj.view(B, T, self.num_heads, self.head_dim)

        # Calculate attention scores
        # Query: [H, D] -> [1, 1, H, D]
        scores = torch.einsum("bthd,hd->bht", x_reshaped, self.query) / (
            self.head_dim**0.5
        )
        attn = F.softmax(scores, dim=-1)

        # Weighted sum: [B, H, T] x [B, T, H, D] -> [B, H, D]
        x_reshaped_t = x_reshaped.permute(0, 2, 1, 3)
        pooled = torch.einsum("bht,bhtd->bhd", attn, x_reshaped_t)
        return pooled.reshape(B, C)


class ConvNeXtAudioClassifier(nn.Module):
    """End-to-End ConvNeXt-1D Audio Classifier with Feature Extraction & Temporal Attention."""

    def __init__(
        self,
        num_classes: int = 35,
        in_channels: int = 80,
        dims: list = [64, 128, 256, 512],
        depths: list = [2, 2, 4, 2],
        sample_rate: int = 16000,
        n_fft: int = 1024,
        hop_length: int = 256,
        n_mels: int = 80,
    ):
        super().__init__()
        self.feature_extractor = AudioFeatureExtractor(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            is_training=True,
        )

        # Stem: maps input Mel spectrogram to first stage dim
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, dims[0], kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(1, dims[0]),
        )

        # Stages with downsampling
        self.stages = nn.ModuleList()
        for i in range(len(dims)):
            stage = nn.Sequential(*[ConvNeXt1DBlock(dims[i]) for _ in range(depths[i])])
            self.stages.append(stage)
            if i < len(dims) - 1:
                # Downsample layer
                downsample = nn.Sequential(
                    nn.GroupNorm(1, dims[i]),
                    nn.Conv1d(dims[i], dims[i + 1], kernel_size=2, stride=2),
                )
                self.stages.append(downsample)

        # Attention Pooling & Classification Head
        self.pool = MultiHeadTemporalAttentionPooling(dims[-1], num_heads=4)
        self.head = nn.Sequential(
            nn.LayerNorm(dims[-1]), nn.Dropout(0.2), nn.Linear(dims[-1], num_classes)
        )

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        # waveforms: [B, 1, Samples]
        mels = self.feature_extractor(waveforms)
        x = self.stem(mels)

        for layer in self.stages:
            x = layer(x)

        x_pooled = self.pool(x)
        logits = self.head(x_pooled)
        return logits

## 10. Model Instantiation and Parameter Summary

Instantiate the 1D ConvNeXt model on the allocated compute device, conduct a parameter audit, and verify tensor shape transitions through a dummy forward pass.

In [ ]:
model = ConvNeXtAudioClassifier(
    num_classes=NUM_CLASSES,
    in_channels=N_MELS,
    dims=[64, 128, 256, 384],
    depths=[2, 2, 4, 2],
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Summary: ConvNeXtAudioClassifier")
print(f"Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB)")
print(f"Trainable Parameters: {trainable_params:,}")

# Verification forward pass
dummy_input = torch.randn(4, 1, TARGET_LENGTH, device=device)
with torch.no_grad():
    dummy_out = model(dummy_input)
print(f"Forward Pass Output Shape: {dummy_out.shape} -> Expected [4, {NUM_CLASSES}]")
assert dummy_out.shape == (4, NUM_CLASSES), "Shape mismatch!"

## 11. Loss Function and Learning Rate Schedule

Initialize label-smoothed Cross-Entropy loss ($\epsilon=0.10$), `AdamW` optimizer (weight decay $10^{-4}$), and `OneCycleLR` dynamic learning rate schedule with cosine annealing over the training duration.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    total_steps=total_steps,
    pct_start=0.15,
    anneal_strategy="cos",
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None


def evaluate(model, dataloader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for waveforms, labels in dataloader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(waveforms)
                loss = criterion(logits, labels)

            total_loss += loss.item() * waveforms.size(0)
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    return (
        total_loss / total,
        (correct / total) * 100.0,
        np.array(all_preds),
        np.array(all_targets),
    )

## 12. Training and Validation Loop Execution

Execute the optimization loop across mini-batches with mixed-precision arithmetic (`torch.amp`), gradient norm clipping ($5.0$), per-epoch validation evaluation, telemetry logging, and model checkpoint serialization.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = 0.0

print(f"Starting training for {EPOCHS} epochs on {device_name}...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(waveforms)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()
            if scale_before <= scale_after:
                scheduler.step()
        else:
            logits = model(waveforms)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            if is_xla:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()
            scheduler.step()

        batch_size = waveforms.size(0)
        train_loss += loss.item() * batch_size
        preds = logits.argmax(dim=-1)
        train_correct += (preds == labels).sum().item()
        train_total += batch_size

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100.0
    epoch_val_loss, epoch_val_acc, _, _ = evaluate(model, val_loader, device)

    current_lr = scheduler.get_last_lr()[0]
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    history["lr"].append(current_lr)

    writer.add_scalar("Loss/Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val", epoch_val_loss, epoch)
    writer.add_scalar("Accuracy/Train", epoch_train_acc, epoch)
    writer.add_scalar("Accuracy/Val", epoch_val_acc, epoch)
    writer.add_scalar("LearningRate", current_lr, epoch)
    writer.flush()

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}% | "
        f"LR: {current_lr:.2e}"
    )

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        save_model_zip(
            model=model,
            metadata={
                "epoch": epoch,
                "val_acc": best_val_acc,
                "classes": COMMAND_CLASSES,
            },
            save_path=MODEL_PATH,
            is_xla=is_xla,
        )
        print(
            f"  --> Saved new best checkpoint to {MODEL_PATH} (Val Acc: {best_val_acc:.2f}%)"
        )

writer.close()
print("Training completed.")

## 13. Quantitative Evaluation, Confusion Analysis, and Error Diagnostics

Evaluate the best-performing model checkpoint on the held-out test split, reporting overall accuracy, Top-3 accuracy, macro-averaged F1 score, and normalized confusion matrix.

In [ ]:
# Load best checkpoint for testing
if MODEL_PATH.exists():
    checkpoint = torch.save if False else torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"Loaded best checkpoint from Epoch {checkpoint['epoch']} with Val Acc: {checkpoint['val_acc']:.2f}%"
    )

test_loss, test_acc, test_preds, test_targets = evaluate(model, test_loader, device)
print(
    f"\nHeld-Out Test Results -> Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.2f}%"
)

# Compute Confusion Matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm_norm,
    annot=False,
    cmap="Blues",
    xticklabels=COMMAND_CLASSES,
    yticklabels=COMMAND_CLASSES,
)
plt.title(
    f"Speech Commands Confusion Matrix (Test Accuracy: {test_acc:.2f}%)", fontsize=14
)
plt.xlabel("Predicted Keyword", fontsize=12)
plt.ylabel("True Keyword", fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Classification Report Summary
report = classification_report(
    test_targets, test_preds, target_names=COMMAND_CLASSES, output_dict=True
)
df_report = pd.DataFrame(report).transpose()
print("Top 10 Performing Classes:")
print(
    df_report.sort_values(by="f1-score", ascending=False).head(10)[
        ["precision", "recall", "f1-score", "support"]
    ]
)

## 14. Interactive Audio Inference and Prediction Pipeline

Implement an end-to-end inference function for arbitrary speech command recordings with Top-$K$ class probabilities and waveform/spectrogram visualization.

In [ ]:
def predict_audio(waveform: torch.Tensor, sample_rate: int, top_k: int = 5):
    """Runs end-to-end inference on a single audio waveform."""
    model.eval()
    if sample_rate != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)

    waveform_norm = pad_or_trim(waveform, TARGET_LENGTH).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(waveform_norm)
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()

    top_indices = np.argsort(probs)[::-1][:top_k]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    # Waveform
    ax1.plot(
        np.linspace(0, 1.0, TARGET_LENGTH),
        waveform_norm[0, 0].cpu().numpy(),
        color="#2ca02c",
    )
    ax1.set_title("Input Audio Waveform")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True, alpha=0.3)

    # Top-K Probabilities
    top_labels = [idx_to_label[i] for i in top_indices]
    top_probs = [probs[i] * 100 for i in top_indices]
    bars = ax2.barh(top_labels[::-1], top_probs[::-1], color="#1f77b4")
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("Probability (%)")
    ax2.set_title(f"Top-{top_k} Predictions")
    for bar in bars:
        ax2.text(
            bar.get_width() + 1,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%",
            va="center",
        )

    plt.tight_layout()
    plt.show()

    print(f"Top Prediction: '{top_labels[0]}' ({top_probs[0]:.2f}%)")
    return ipd.Audio(waveform_norm[0, 0].cpu().numpy(), rate=SAMPLE_RATE)


# Test inference on a random sample from test_set
rand_idx = random.randint(0, len(test_set) - 1)
test_wave, sr, true_label, _, _ = test_set[rand_idx]
print(f"Testing Sample Index {rand_idx} | Ground Truth: '{true_label}'")
predict_audio(test_wave, sr)